In [1]:
from torch.utils.data import Dataset, DataLoader
import torch
import pandas as pd
import numpy as np
import pickle

In [2]:
device ="mps"

In [3]:
with open('../../data/complete_features.pkl', 'rb') as f:
    all_data = pickle.load(f)

In [4]:
data = all_data["scaled_featured"].copy()
data["close"] = all_data["Close"]

In [5]:
# log returns
data["log_return"] = np.log(data["close"] / data["close"].shift(1))
data.dropna(inplace=True)

In [6]:
data

,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,vol_slope_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20,close,log_return
Date,,,,,,,,,,,,,,
2008-07-08,-1.211754,-1.882225,2.077736,-0.022097,-1.117078,-1.046961,-0.542321,-2.455523,1.450259,0.981827,-2.206775,-0.513460,3988.550049,-0.010339
2008-07-09,-0.957426,-1.882225,2.289506,1.636228,-1.218247,-0.338846,-0.561373,-1.988903,1.476862,0.785786,-1.960992,-0.565134,4157.100098,0.041390
2008-07-10,-0.975703,-1.882225,2.287995,-0.006124,-1.013921,-0.474801,-0.596656,-2.401313,1.493754,0.505013,-1.940271,-0.562557,4162.200195,0.001226
2008-07-11,-1.136144,-1.882225,2.391761,0.804554,-1.050675,-0.396400,-0.475448,-2.987661,1.507027,0.400347,-2.078101,-0.510752,4049.000000,-0.027574
2008-07-14,-1.257924,-1.882225,2.371776,-0.148393,-1.268721,-0.518601,-0.436624,-3.173512,1.521334,0.430252,-2.071028,-0.455743,4039.699951,-0.002300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-15,0.375855,0.654743,-0.795604,0.021503,0.541634,-0.126723,-0.727148,-1.189892,-0.988672,-0.309058,0.314003,0.507783,25069.199219,-0.001785
2025-09-16,0.280399,0.654743,-0.833663,-0.287568,0.045456,-0.238463,-0.708679,-1.276291,-1.000565,-0.327333,0.453848,0.513354,25239.099609,0.006754
2025-09-17,0.260114,0.654743,-0.836851,-0.019044,0.154682,-0.363408,-0.703390,-1.335486,-1.009414,-0.239281,0.522324,0.511624,25330.250000,0.003605


In [7]:
close = data["close"]
log_return = data["log_return"]

In [8]:
data.shape

(4218, 14)

In [9]:
data.drop(columns=["close","log_return"], inplace=True)

In [10]:
data

,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,vol_slope_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20
Date,,,,,,,,,,,,
2008-07-08,-1.211754,-1.882225,2.077736,-0.022097,-1.117078,-1.046961,-0.542321,-2.455523,1.450259,0.981827,-2.206775,-0.513460
2008-07-09,-0.957426,-1.882225,2.289506,1.636228,-1.218247,-0.338846,-0.561373,-1.988903,1.476862,0.785786,-1.960992,-0.565134
2008-07-10,-0.975703,-1.882225,2.287995,-0.006124,-1.013921,-0.474801,-0.596656,-2.401313,1.493754,0.505013,-1.940271,-0.562557
2008-07-11,-1.136144,-1.882225,2.391761,0.804554,-1.050675,-0.396400,-0.475448,-2.987661,1.507027,0.400347,-2.078101,-0.510752
2008-07-14,-1.257924,-1.882225,2.371776,-0.148393,-1.268721,-0.518601,-0.436624,-3.173512,1.521334,0.430252,-2.071028,-0.455743
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-15,0.375855,0.654743,-0.795604,0.021503,0.541634,-0.126723,-0.727148,-1.189892,-0.988672,-0.309058,0.314003,0.507783
2025-09-16,0.280399,0.654743,-0.833663,-0.287568,0.045456,-0.238463,-0.708679,-1.276291,-1.000565,-0.327333,0.453848,0.513354
2025-09-17,0.260114,0.654743,-0.836851,-0.019044,0.154682,-0.363408,-0.703390,-1.335486,-1.009414,-0.239281,0.522324,0.511624


In [11]:
class TimeSeriesWindowDataset(Dataset):
    def __init__(self, data, window_size=60):
        """
        data: numpy array [T, D]
        """
        self.data = torch.tensor(data, dtype=torch.float32)
        self.window_size = window_size

    def __len__(self):
        return len(self.data) - self.window_size + 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.window_size]
        return x

window_size = 60
val_size = 0

train_size = int(len(data) * (1 - val_size))
train_data = data.iloc[:train_size]
val_data = data.iloc[train_size:]

train_dataset = TimeSeriesWindowDataset(train_data.values, window_size)
val_dataset = TimeSeriesWindowDataset(val_data.values, window_size)
combined_dataset = TimeSeriesWindowDataset(data.values, window_size)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
combined_loader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=False)



In [12]:
import torch.nn as nn
import torch.nn.functional as F

class TS2VecModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=5):
        super(TS2VecModel, self).__init__()
        layers = []
        for i in range(num_layers):
            dilation = 2 ** i
            layers.append(nn.Conv1d(input_dim if i == 0 else hidden_dim, hidden_dim, kernel_size=3, padding=dilation, dilation=dilation))
            layers.append(nn.ReLU())
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # x: [B, T, D]
        x = x.permute(0, 2, 1)  # [B, D, T]
        x = self.network(x)     # [B, H, T]
        x = x.permute(0, 2, 1)  # [B, T, H]
        # normalize
        x = F.normalize(x, p=2, dim=-1)
        return x
    
model = TS2VecModel(input_dim=data.shape[1], hidden_dim=128, num_layers=6).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [13]:
def time_mask(x, mask_ratio=0.2):
    B, T, D = x.shape
    mask_len = int(T * mask_ratio)
    start = np.random.randint(0, T - mask_len)
    x = x.clone()
    x[:, start:start+mask_len, :] = 0
    return x


def jitter(x, sigma=0.02):
    return x + sigma * torch.randn_like(x)


def temporal_pooling(z):
    if z.size(1) % 2 == 1:
        z = z[:, :-1]
    return z.reshape(z.size(0), z.size(1)//2, 2, z.size(2)).mean(dim=2)


def ts2vec_contrastive_loss_vectorized(
    z1, z2,
    temperature=0.1,
    base_exclusion_radius=5
):
    """
    z1, z2: [B, T, C]
    """
    B, T, C = z1.shape
    device = z1.device

    # flatten (instance, time)
    z1_flat = z1.reshape(B*T, C)
    z2_flat = z2.reshape(B*T, C)

    # cosine similarity == dot product because embeddings are normalized
    sim = torch.matmul(z1_flat, z2_flat.T) / temperature   # [BT, BT]

    # ----- temporal negative mask -----
    exclusion_radius = min(base_exclusion_radius, (T - 1) // 2)

    if exclusion_radius == 0:
        return torch.tensor(0.0, device=device)

    # time index per row
    time_idx = torch.arange(T, device=device).repeat(B)     # [BT]

    # batch index per row
    batch_idx = torch.arange(B, device=device).repeat_interleave(T)

    # same batch & temporally close → mask out
    temporal_dist = torch.abs(time_idx[:, None] - time_idx[None, :])
    same_batch = batch_idx[:, None] == batch_idx[None, :]

    invalid_negatives = same_batch & (temporal_dist <= exclusion_radius)

    # allow diagonal (positive pairs)
    diag = torch.eye(B*T, device=device, dtype=torch.bool)
    invalid_negatives = invalid_negatives & (~diag)

    # mask invalid negatives
    sim = sim.masked_fill(invalid_negatives, -1e9)

    # positives are diagonal
    labels = torch.arange(B*T, device=device)

    return F.cross_entropy(sim, labels)



def hierarchical_ts2vec_loss_v2(
    z1, z2,
    temperature=0.1,
    exclusion_radius=5,
    min_time=2
):
    total_loss = 0.0
    depth = 0

    while z1.size(1) >= min_time:
        T = z1.size(1)

        if T > 2 * exclusion_radius + 1:
            total_loss += ts2vec_contrastive_loss_vectorized(
                z1, z2,
                temperature,
                exclusion_radius
            )
            depth += 1

        z1 = temporal_pooling(z1)
        z2 = temporal_pooling(z2)

    return total_loss / max(depth, 1)



In [14]:
def temporal_contrast_score(Z, k=20):
    """
    This function computes the temporal contrast score for a given set of embeddings Z.
    Z: [T, C] numpy array of embeddings
    k: temporal gap for negative pairs
    """
    pos = []
    neg = []
    
    for i in range(len(Z) - k - 1):
        pos.append(np.dot(Z[i], Z[i+1]))
        neg.append(np.dot(Z[i], Z[i+k]))
    
    return np.mean(pos) - np.mean(neg)

# score = temporal_contrast_score(embeddings_full)
def temporal_contrast_score_multi(Z, pos_gap=1, neg_gaps=(10,20,40)):
    pos = []
    neg = []

    for i in range(len(Z) - max(neg_gaps) - 1):
        pos.append(np.dot(Z[i], Z[i+pos_gap]))
        for k in neg_gaps:
            neg.append(np.dot(Z[i], Z[i+k]))

    return np.mean(pos) - np.mean(neg)


from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

def predictive_score(Z, forward_returns):
    X = Z[:-1]
    y = forward_returns[1:]

    model = Ridge(alpha=1.0)
    model.fit(X, y)
    y_pred = model.predict(X)

    return r2_score(y, y_pred)

In [15]:
from dataclasses import dataclass

@dataclass
class WindowParams:
    window_size: int

@dataclass
class ArchitectureParams:
    hidden_dim: int
    num_layers: int

@dataclass
class ContrastiveParams:
    temperature: float
    exclusion_radius: int

@dataclass
class AugmentationParams:
    mask_ratio: float
    jitter_sigma: float

@dataclass
class OptimizationParams:
    learning_rate: float
    batch_size: int

In [34]:

long={'window': {'window_size': 180},
 'architecture': {'hidden_dim': 256, 'num_layers': 4},
 'contrastive': {'temperature': 0.10901392823581604, 'exclusion_radius': 18},
 'augmentation': {'mask_ratio': 0.10112014206194599,
  'jitter_sigma': 0.019884546877356246},
 'optimization': {'learning_rate': 0.0012543308801674451, 'batch_size': 64}}


short = {'window': 60,
 'architecture': {'hidden_dim': 128, 'num_layers': 4},
 'contrastive': {'temperature': 0.09418251159949012, 'exclusion_radius': 3},
 'augmentation': {'mask_ratio': 0.1153883102823196,
  'jitter_sigma': 0.006324426296094188},
 'optimization': {'learning_rate': 0.0017009936483370917, 'batch_size': 32}}



long_term_model_hyperparams_config = {
    "window_size": [30, 180, 30],  # range for window size  
    "hidden_dim": [ 128, 256,512],    
    "num_layers": [3, 10],
    "temperature": [0.05, 0.5],
    "exclusion_radius": [2, 20],
    "mask_ratio": [0.1, 0.5],
    "jitter_sigma": [0.005, 0.05],
    "learning_rate": [1e-4, 3e-3],
    "batch_size": [32,64,128]
}

short_term_model_hyperparams_config = {
    "window_size": [40, 80, 20],  # range for window size
    "hidden_dim": [32, 64, 128],
    "num_layers": [2, 7],
    "temperature": [0.05, 0.5],
    "exclusion_radius": [1, 5],
    "mask_ratio": [0.1, 0.5],
    "jitter_sigma": [0.005, 0.05],
    "learning_rate": [1e-4, 3e-3],
    "batch_size": [16,32,64]
}

mode = "long_term"  # or "short_term"

if mode == "long_term":
    hyperparams_config = long_term_model_hyperparams_config
else:
    hyperparams_config = short_term_model_hyperparams_config

In [35]:
import optuna
import torch
import numpy as np

class BaseTS2VecObjective:
    def __init__(self, data, device, fixed_params):
        self.data = data
        self.device = device
        self.fixed_params = fixed_params  # dict of params from previous stages

    def build_model(self, arch_params):
        return TS2VecModel(
            input_dim=self.data.shape[1],
            hidden_dim=arch_params.hidden_dim,
            num_layers=arch_params.num_layers
        ).to(self.device)

    def compute_score(self, model):
        model.eval()
        with torch.no_grad():
            full_tensor = torch.tensor(self.data.values, dtype=torch.float32).unsqueeze(0).to(self.device)
            z = model(full_tensor).squeeze(0).cpu().numpy()
            # use forward returns for predictive score
            return predictive_score(z, log_return.values)

    def train_model(self, model, train_loader, contrastive_params, aug_params, opt_params, epochs=3):
        optimizer = torch.optim.Adam(model.parameters(), lr=opt_params.learning_rate)

        model.train()
        for _ in range(epochs):  # small epochs for HPO
            for x in train_loader:
                x = x.to(self.device)
                x1 = jitter(time_mask(x, aug_params.mask_ratio), aug_params.jitter_sigma)
                x2 = jitter(time_mask(x, aug_params.mask_ratio), aug_params.jitter_sigma)

                z1 = model(x1)
                z2 = model(x2)

                loss = hierarchical_ts2vec_loss_v2(
                    z1, z2,
                    temperature=contrastive_params.temperature,
                    exclusion_radius=contrastive_params.exclusion_radius
                )

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        return model
    


In [36]:
def create_loader(data, window_size, batch_size):
    dataset = TimeSeriesWindowDataset(data.values, window_size)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


In [37]:

class WindowStageObjective(BaseTS2VecObjective):

    def __call__(self, trial):
        window_size = trial.suggest_int("window_size", hyperparams_config["window_size"][0] , hyperparams_config["window_size"][1], step=hyperparams_config["window_size"][2])

        window_params = WindowParams(window_size)

        train_loader = create_loader(self.data, window_size, batch_size=64)

        # fixed architecture for stage 0
        arch_params = ArchitectureParams(hidden_dim=128, num_layers=5)
        contrastive_params = ContrastiveParams(0.1, 5)
        aug_params = AugmentationParams(0.2, 0.02)
        opt_params = OptimizationParams(1e-3, 64)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("window_params", window_params.__dict__)

        return score

In [38]:
type(data.values)

numpy.ndarray

In [39]:
study_window = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="window_stage",
    load_if_exists=True
)

# suggest optimal trail count based on combination of parameters 


study_window.optimize(WindowStageObjective(data, device, {}), n_trials=7)

[I 2026-02-27 18:00:08,211] A new study created in RDB with name: window_stage
[I 2026-02-27 18:00:12,674] Trial 0 finished with value: 0.050904186416149466 and parameters: {'window_size': 90}. Best is trial 0 with value: 0.050904186416149466.
[I 2026-02-27 18:00:14,331] Trial 1 finished with value: 0.042956159312403486 and parameters: {'window_size': 30}. Best is trial 0 with value: 0.050904186416149466.
[I 2026-02-27 18:00:29,307] Trial 2 finished with value: 0.05436669860978349 and parameters: {'window_size': 180}. Best is trial 2 with value: 0.05436669860978349.
[I 2026-02-27 18:00:36,844] Trial 3 finished with value: 0.04943881063751843 and parameters: {'window_size': 120}. Best is trial 2 with value: 0.05436669860978349.
[I 2026-02-27 18:00:38,963] Trial 4 finished with value: 0.04542773359364405 and parameters: {'window_size': 60}. Best is trial 2 with value: 0.05436669860978349.
[I 2026-02-27 18:00:40,372] Trial 5 finished with value: 0.050687018414122376 and parameters: {'wind

In [40]:
study_window.best_trial.user_attrs["window_params"]

{'window_size': 180}

In [41]:
class ArchitectureStageObjective(BaseTS2VecObjective):

    def __call__(self, trial):

        hidden_dim = trial.suggest_categorical("hidden_dim", hyperparams_config["hidden_dim"])
        num_layers = trial.suggest_int("num_layers", hyperparams_config["num_layers"][0], hyperparams_config["num_layers"][1])

        arch_params = ArchitectureParams(hidden_dim, num_layers)

        window_size = self.fixed_params["window_size"]

        train_loader = create_loader(self.data, window_size, batch_size=64)

        contrastive_params = ContrastiveParams(0.1, 5)
        aug_params = AugmentationParams(0.2, 0.02)
        opt_params = OptimizationParams(1e-3, 64)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("arch_params", arch_params.__dict__)

        return score
    
#!/usr/bin/env python
import os

# Delete the existing study to avoid "dynamic value space" error
if os.path.exists("ts2vec_hpo.db"):
    try:
        optuna.delete_study(study_name="arch_stage", storage="sqlite:///ts2vec_hpo.db")
    except:
        pass

study_arch = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="arch_stage",
    load_if_exists=True
)


study_arch.optimize(ArchitectureStageObjective(data, device, {**study_window.best_trial.user_attrs["window_params"]} ), n_trials= 30)

[I 2026-02-27 18:01:20,401] A new study created in RDB with name: arch_stage
[I 2026-02-27 18:01:34,643] Trial 0 finished with value: 0.06108141168901726 and parameters: {'hidden_dim': 128, 'num_layers': 3}. Best is trial 0 with value: 0.06108141168901726.
[I 2026-02-27 18:01:48,681] Trial 1 finished with value: 0.0620859089728103 and parameters: {'hidden_dim': 128, 'num_layers': 3}. Best is trial 1 with value: 0.0620859089728103.
[I 2026-02-27 18:02:03,520] Trial 2 finished with value: 0.0496431994407468 and parameters: {'hidden_dim': 128, 'num_layers': 7}. Best is trial 1 with value: 0.0620859089728103.
[I 2026-02-27 18:02:18,113] Trial 3 finished with value: 0.059232582800011535 and parameters: {'hidden_dim': 128, 'num_layers': 4}. Best is trial 1 with value: 0.0620859089728103.
[I 2026-02-27 18:03:05,487] Trial 4 finished with value: 0.05454666849887868 and parameters: {'hidden_dim': 512, 'num_layers': 10}. Best is trial 1 with value: 0.0620859089728103.
[I 2026-02-27 18:03:30,954]

In [42]:
# Contrastive stage:

# temperature = trial.suggest_float("temperature", 0.05, 0.5, log=True)
# exclusion_radius = trial.suggest_int("exclusion_radius", 2, 20)

class ContrastiveStageObjective(BaseTS2VecObjective):

    def __call__(self, trial):

        temperature = trial.suggest_float("temperature", hyperparams_config["temperature"][0], hyperparams_config["temperature"][1], log=True)
        exclusion_radius = trial.suggest_int("exclusion_radius", hyperparams_config["exclusion_radius"][0], hyperparams_config["exclusion_radius"][1])

        contrastive_params = ContrastiveParams(temperature, exclusion_radius)

        window_size = self.fixed_params["window_size"]
        arch_params = ArchitectureParams(self.fixed_params["hidden_dim"], self.fixed_params["num_layers"])

        train_loader = create_loader(self.data, window_size, batch_size=64)

        aug_params = AugmentationParams(0.2, 0.02)
        opt_params = OptimizationParams(1e-3, 64)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("contrastive_params", contrastive_params.__dict__)

        return score
    
study_contrastive = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="contrastive_stage",
    load_if_exists=True
)

study_contrastive.optimize(ContrastiveStageObjective(data, device, 
                                                     {
    **study_window.best_trial.user_attrs["window_params"],
    **study_arch.best_trial.user_attrs["arch_params"]
}
), n_trials=50)

[I 2026-02-27 18:19:21,568] A new study created in RDB with name: contrastive_stage
[I 2026-02-27 18:19:38,462] Trial 0 finished with value: 0.07622515662826701 and parameters: {'temperature': 0.12843987805219234, 'exclusion_radius': 16}. Best is trial 0 with value: 0.07622515662826701.
[I 2026-02-27 18:19:55,416] Trial 1 finished with value: 0.07346426298509412 and parameters: {'temperature': 0.1198879184555883, 'exclusion_radius': 9}. Best is trial 0 with value: 0.07622515662826701.
[I 2026-02-27 18:20:12,417] Trial 2 finished with value: 0.03337804235606645 and parameters: {'temperature': 0.25097782258054363, 'exclusion_radius': 5}. Best is trial 0 with value: 0.07622515662826701.
[I 2026-02-27 18:20:29,502] Trial 3 finished with value: 0.081651982700782 and parameters: {'temperature': 0.1015637747023826, 'exclusion_radius': 5}. Best is trial 3 with value: 0.081651982700782.
[I 2026-02-27 18:20:46,624] Trial 4 finished with value: 0.04119563418671501 and parameters: {'temperature': 

In [43]:
# Augmentation stage:

# mask_ratio = trial.suggest_float("mask_ratio", 0.1, 0.4)
# jitter_sigma = trial.suggest_float("jitter_sigma", 0.005, 0.05, log=True)

class AugmentationStageObjective(BaseTS2VecObjective):
    
    def __call__(self, trial):

        mask_ratio = trial.suggest_float("mask_ratio", hyperparams_config["mask_ratio"][0], hyperparams_config["mask_ratio"][1])
        jitter_sigma = trial.suggest_float("jitter_sigma", hyperparams_config["jitter_sigma"][0], hyperparams_config["jitter_sigma"][1], log=True)

        aug_params = AugmentationParams(mask_ratio, jitter_sigma)

        window_size = self.fixed_params["window_size"]
        arch_params = ArchitectureParams(self.fixed_params["hidden_dim"], self.fixed_params["num_layers"])
        contrastive_params = ContrastiveParams(self.fixed_params["temperature"], self.fixed_params["exclusion_radius"])

        train_loader = create_loader(self.data, window_size, batch_size=64)

        opt_params = OptimizationParams(1e-3, 64)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("aug_params", aug_params.__dict__)

        return score
    
study_aug = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="augmentation_stage",
    load_if_exists=True
)
study_aug.optimize(AugmentationStageObjective(data, device, 
                                                     {
      **study_window.best_trial.user_attrs["window_params"],
    **study_arch.best_trial.user_attrs["arch_params"],
    **study_contrastive.best_trial.user_attrs["contrastive_params"]
}
), n_trials=50)

[I 2026-02-27 22:12:10,765] A new study created in RDB with name: augmentation_stage
[I 2026-02-27 22:12:28,005] Trial 0 finished with value: 0.10942956160620576 and parameters: {'mask_ratio': 0.2857537118458853, 'jitter_sigma': 0.029130636108840445}. Best is trial 0 with value: 0.10942956160620576.
[I 2026-02-27 22:12:44,793] Trial 1 finished with value: 0.10141577823095138 and parameters: {'mask_ratio': 0.19045612026580555, 'jitter_sigma': 0.00793790567271901}. Best is trial 0 with value: 0.10942956160620576.
[I 2026-02-27 22:13:01,610] Trial 2 finished with value: 0.12030535725413294 and parameters: {'mask_ratio': 0.24846365142893945, 'jitter_sigma': 0.029822553387346386}. Best is trial 2 with value: 0.12030535725413294.
[I 2026-02-27 22:13:18,560] Trial 3 finished with value: 0.09797382982022795 and parameters: {'mask_ratio': 0.43428330423072564, 'jitter_sigma': 0.007993878054723123}. Best is trial 2 with value: 0.12030535725413294.
[I 2026-02-27 22:13:35,727] Trial 4 finished with

In [44]:
# Optimization stage:

# learning_rate = trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True)
# batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

class OptimizationStageObjective(BaseTS2VecObjective):
    
    def __call__(self, trial):

        learning_rate = trial.suggest_float("learning_rate", hyperparams_config["learning_rate"][0], hyperparams_config["learning_rate"][1], log=True)
        batch_size = trial.suggest_categorical("batch_size", hyperparams_config["batch_size"])

        opt_params = OptimizationParams(learning_rate, batch_size)

        window_size = self.fixed_params["window_size"]
        arch_params = ArchitectureParams(self.fixed_params["hidden_dim"], self.fixed_params["num_layers"])
        contrastive_params = ContrastiveParams(self.fixed_params["temperature"], self.fixed_params["exclusion_radius"])
        aug_params = AugmentationParams(self.fixed_params["mask_ratio"], self.fixed_params["jitter_sigma"])

        train_loader = create_loader(self.data, window_size, batch_size)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("opt_params", opt_params.__dict__)

        return score
    
study_opt = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="optimization_stage",
    load_if_exists=True
)
study_opt.optimize(OptimizationStageObjective(data, device,
                                                     {
  **study_window.best_trial.user_attrs["window_params"],
    **study_arch.best_trial.user_attrs["arch_params"],
    **study_contrastive.best_trial.user_attrs["contrastive_params"],
    **study_aug.best_trial.user_attrs["aug_params"]
}), n_trials=50)

[I 2026-02-27 22:28:42,627] A new study created in RDB with name: optimization_stage


[I 2026-02-27 22:28:52,277] Trial 0 finished with value: 0.13562347112467787 and parameters: {'learning_rate': 0.00016014997535220753, 'batch_size': 32}. Best is trial 0 with value: 0.13562347112467787.
[I 2026-02-27 22:29:26,347] Trial 1 finished with value: 0.1357538896477507 and parameters: {'learning_rate': 0.000384636048571105, 'batch_size': 128}. Best is trial 1 with value: 0.1357538896477507.
[I 2026-02-27 22:30:00,437] Trial 2 finished with value: 0.13921666219515794 and parameters: {'learning_rate': 0.0003866680481593282, 'batch_size': 128}. Best is trial 2 with value: 0.13921666219515794.
[I 2026-02-27 22:30:10,163] Trial 3 finished with value: 0.10568016816002579 and parameters: {'learning_rate': 0.0005916444986841068, 'batch_size': 32}. Best is trial 2 with value: 0.13921666219515794.
[I 2026-02-27 22:30:19,712] Trial 4 finished with value: 0.10484409157429442 and parameters: {'learning_rate': 0.0012211065918293872, 'batch_size': 32}. Best is trial 2 with value: 0.139216662

In [ ]:
from datetime import datetime

class TS2VecHyperparameterPipeline:
    def __init__(self, data, device, mode="long_term"):
        self.data = data
        self.device = device
        self.results = {}
        self.mode = mode  # "long_term" or "short_term"

    def _run_stage(self, name, objective_cls, fixed_params, n_trials, result_key, attr_key):
        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(),
            storage="sqlite:///ts2vec_hpo.db",
            study_name=f"{name}_stage_{self.mode}_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
            load_if_exists=True,
        )
        study.optimize(objective_cls(self.data, self.device, fixed_params), n_trials=n_trials)
        self.results[result_key] = study.best_trial.user_attrs[attr_key]

    def run(self):
        self._run_stage(
            "window",
            WindowStageObjective,
            {},
            n_trials=7,
            result_key="window",
            attr_key="window_params",
        )
        self._run_stage(
            "arch",
            ArchitectureStageObjective,
            self.results["window"],
            n_trials=30,
            result_key="architecture",
            attr_key="arch_params",
        )
        self._run_stage(
            "contrastive",
            ContrastiveStageObjective,
            {**self.results["window"], **self.results["architecture"]},
            n_trials=50,
            result_key="contrastive",
            attr_key="contrastive_params",
        )
        self._run_stage(
            "augmentation",
            AugmentationStageObjective,
            {**self.results["window"], **self.results["architecture"], **self.results["contrastive"]},
            n_trials=50,
            result_key="augmentation",
            attr_key="aug_params",
        )
        self._run_stage(
            "optimization",
            OptimizationStageObjective,
            {
                **self.results["window"],
                **self.results["architecture"],
                **self.results["contrastive"],
                **self.results["augmentation"],
            },
            n_trials=50,
            result_key="optimization",
            attr_key="opt_params",
        )
        return self.results

# Run the pipeline
pipeline = TS2VecHyperparameterPipeline(data, device, mode="long_term")
final_results = pipeline.run()
print(final_results)

In [45]:
# list all the best parameters from each stage
param_summary = {
    "window": study_window.best_trial.user_attrs["window_params"],
    "architecture": study_arch.best_trial.user_attrs["arch_params"],
    "contrastive": study_contrastive.best_trial.user_attrs["contrastive_params"],
    "augmentation": study_aug.best_trial.user_attrs["aug_params"],
    "optimization": study_opt.best_trial.user_attrs["opt_params"]
}

```json
{'window': {'window_size': 180},
 'architecture': {'hidden_dim': 256, 'num_layers': 4},
 'contrastive': {'temperature': 0.10901392823581604, 'exclusion_radius': 18},
 'augmentation': {'mask_ratio': 0.10112014206194599,
  'jitter_sigma': 0.019884546877356246},
 'optimization': {'learning_rate': 0.0012543308801674451, 'batch_size': 64}}
```
```json
{'window': 60,
 'architecture': {'hidden_dim': 128, 'num_layers': 4},
 'contrastive': {'temperature': 0.09418251159949012, 'exclusion_radius': 3},
 'augmentation': {'mask_ratio': 0.1153883102823196,
  'jitter_sigma': 0.006324426296094188},
 'optimization': {'learning_rate': 0.0017009936483370917, 'batch_size': 32}}
```
 

In [33]:
param_summary

{'window': {'window_size': 40},
 'architecture': {'hidden_dim': 128, 'num_layers': 3},
 'contrastive': {'temperature': 0.05865611450479419, 'exclusion_radius': 5},
 'augmentation': {'mask_ratio': 0.11860240054087702,
  'jitter_sigma': 0.028089550039277756},
 'optimization': {'learning_rate': 0.00015072787810725861, 'batch_size': 32}}

In [46]:
param_summary

{'window': {'window_size': 180},
 'architecture': {'hidden_dim': 256, 'num_layers': 3},
 'contrastive': {'temperature': 0.06295392144086782, 'exclusion_radius': 19},
 'augmentation': {'mask_ratio': 0.4379409890039726,
  'jitter_sigma': 0.045869891350296864},
 'optimization': {'learning_rate': 0.00011624440419822255, 'batch_size': 64}}